In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# NLTK Kneser-Ney interpolated (Hindle et al. ICSE 2012)
from nltk.lm import KneserNeyInterpolated, Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_sequence
from nltk.lm.vocabulary import Vocabulary
from nltk.util import ngrams

# Same filtering as a.ipynb
df = pd.read_csv("../../dataset/data/final_dataset_new_corrected.csv")
for col in ["doc_entropy", "doc_code_overlap", "doc_redundancy"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["doc_entropy", "doc_code_overlap", "doc_redundancy"])

agent_docs     = df[df["group"] == "agent"]["doc_text"].astype(str).tolist()
developer_docs = df[df["group"] == "human"]["doc_text"].astype(str).tolist()

print(f"Agent docs:     {len(agent_docs)}")
print(f"Developer docs: {len(developer_docs)}")

Agent docs:     3744
Developer docs: 2177


In [7]:
# Words appearing fewer than UNK_CUTOFF times in training are replaced with <UNK>.
# This ensures <UNK> has a non-zero count in training so test OOV words get
# a finite probability rather than zero (which would make entropy = inf).
UNK       = '<UNK>'
UNK_CUTOFF = 2


def train_lm(n, docs):
    """
    Train an order-n language model on whitespace-tokenized docs.
    Uses Kneser-Ney interpolated smoothing for n >= 2 (Hindle et al.);
    falls back to Laplace for n=1 since KN requires bigram context counts.
    """
    tokenized = [d.split() for d in docs if d.split()]

    # Build a vocabulary; mark words below cutoff as <UNK>
    all_words  = [w for sent in tokenized for w in sent]
    train_vocab = Vocabulary(all_words, unk_cutoff=UNK_CUTOFF)

    # Replace rare words with <UNK> in training sequences so the model
    # assigns a non-zero probability to <UNK> at test time
    tokenized_unk = [
        [w if w in train_vocab else UNK for w in sent]
        for sent in tokenized
    ]

    train_data, fitted_vocab = padded_everygram_pipeline(n, tokenized_unk)
    model = KneserNeyInterpolated(n) if n > 1 else Laplace(n)
    model.fit(train_data, fitted_vocab)
    return model, train_vocab


def doc_cross_entropy(model, train_vocab, doc, n):
    """
    Per-sample cross-entropy (bits) of a single doc under the given model.
    Matches Hindle et al.: log2, padded n-grams, mean over n-gram positions.
    """
    tokens = doc.split()
    if not tokens:
        return np.nan
    # Map OOV test tokens to <UNK> using the training vocabulary
    tokens = [w if w in train_vocab else UNK for w in tokens]
    padded = list(pad_sequence(
        tokens, n,
        pad_left=True,  left_pad_symbol='<s>',
        pad_right=True, right_pad_symbol='</s>',
    ))
    test_ngrams = list(ngrams(padded, n))
    if not test_ngrams:
        return np.nan
    try:
        ce = model.entropy(test_ngrams)   # returns bits (log2)
        return ce if np.isfinite(ce) else np.nan
    except (ZeroDivisionError, ValueError):
        return np.nan

In [ ]:
N_SPLITS = 10
records  = []

for n in range(1, 6):
    print(f"n={n}", end=" ", flush=True)
    for label, docs in [("Agent", agent_docs), ("Developer", developer_docs)]:
        arr = np.array(docs, dtype=object)
        kf  = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
        for train_idx, test_idx in kf.split(arr):
            model, train_vocab = train_lm(n, arr[train_idx])
            for doc in arr[test_idx]:
                ce = doc_cross_entropy(model, train_vocab, doc, n)
                if not np.isnan(ce):
                    records.append({"n": n, "Group": label, "cross_entropy": ce})
        print(".", end="", flush=True)
    print(" done")

results = pd.DataFrame(records)
print(f"\nTotal records: {len(results)}")
print(results.groupby(["n", "Group"]).size().unstack())

n=1 .. done
n=2 

In [ ]:
summary = (
    results.groupby(["n", "Group"])["cross_entropy"]
    .agg(mean="mean", std="std")
    .round(4)
)
print("Mean and Std of per-sample cross-entropy (bits) — Kneser-Ney interpolated")
print("=" * 60)
print(summary.to_string())

In [ ]:
def sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

def whisker_top(vals):
    q1, q3 = np.percentile(vals, [25, 75])
    return min(float(np.max(vals)), q3 + 1.5 * (q3 - q1))

palette = {"Agent": "#A7C7E7", "Developer": "#BDE5B8"}

fig, ax = plt.subplots(figsize=(11, 6), dpi=150)

sns.boxplot(
    data=results, x="n", y="cross_entropy", hue="Group",
    palette=palette, showfliers=False, linewidth=1.2,
    order=[1, 2, 3, 4, 5], ax=ax,
)

# seaborn places 2 hues at x-0.2 and x+0.2 (default width=0.8)
y_range  = results["cross_entropy"].quantile(0.95) - results["cross_entropy"].quantile(0.05)
step     = y_range * 0.07
ann_tops = []

for i, n in enumerate(range(1, 6)):
    ag = results[(results["n"] == n) & (results["Group"] == "Agent")]["cross_entropy"].values
    dv = results[(results["n"] == n) & (results["Group"] == "Developer")]["cross_entropy"].values
    _, p = mannwhitneyu(ag, dv, alternative="two-sided")

    x1, x2 = i - 0.2, i + 0.2
    y0      = max(whisker_top(ag), whisker_top(dv)) + step * 0.3
    bar_h   = step * 0.35

    ax.plot([x1, x1, x2, x2], [y0, y0+bar_h, y0+bar_h, y0], lw=1.2, c="k")
    ax.text((x1+x2)/2, y0+bar_h+step*0.05, sig_label(p),
            ha="center", va="bottom", fontsize=12)
    ann_tops.append(y0 + bar_h + step * 0.5)

ax.set_ylim(ax.get_ylim()[0], max(ann_tops) + step * 0.3)

ax.set_xlabel("n-gram order (n)", fontsize=13)
ax.set_ylabel("Per-sample cross-entropy (bits)", fontsize=13)
ax.set_title(
    "N-gram Cross-Entropy: Agent vs. Developer Documentation\n"
    "(Kneser-Ney interpolated, 10-fold CV)",
    fontsize=14, fontweight="bold"
)
ax.legend(title="Group", fontsize=11, title_fontsize=11)
ax.tick_params(axis="both", labelsize=11)
plt.tight_layout()
plt.savefig("ngram_cross_entropy.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nMann-Whitney U p-values (two-sided, uncorrected):")
for n in range(1, 6):
    ag = results[(results["n"] == n) & (results["Group"] == "Agent")]["cross_entropy"].values
    dv = results[(results["n"] == n) & (results["Group"] == "Developer")]["cross_entropy"].values
    _, p = mannwhitneyu(ag, dv, alternative="two-sided")
    print(f"  n={n}: p={p:.4e}  {sig_label(p)}")